# Simple classification

In [ ]:
import matplotlib.pyplot as plt
from functions import extract_features, feat_extract_overall, tune, fit_and_predict, extract_importances, print_results
import os
import numpy as np
import seaborn as sns
import polars as pl
import yaml

from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

balanced_accuracy = True

## Parameters

In [ ]:
# Preprocessing
input_folder = "../cells/gt_label"
files = os.listdir(input_folder)
channels = ['ereg']

# Class names
class_0_str = "no-response"
class_1_str = "any-response"

# load in splits
k_fold_file = "../output/config/k_fold.yaml"

# load yaml
with open(k_fold_file, "r") as ymlfile:
    k_fold_file = yaml.safe_load(ymlfile)

train_folds = k_fold_file["splits"]["train"]
val_folds = k_fold_file["splits"]["val"]
test_folds = k_fold_file["splits"]["test"]

## Feature extraction

In [ ]:
if not os.path.exists("../output/simple_classification"):
    os.makedirs("../output/simple_classification")

features_path = "../output/simple_classification/features.parquet"

if not os.path.exists(features_path):
    features = extract_features(files, input_folder)
    features.write_parquet(features_path)

features = pl.read_parquet(features_path)

## Per-cell shape features

### Global features

In [ ]:
feature_list = [
    "cell_rgyration",
    "cell_linearity",
    "cell_planarity",
    "cell_length",
    "cell_area",
    "cell_perimeter",
    ]

X_train_list, Y_train_list, X_val_list, Y_val_list, X_test_list, Y_test_list = feat_extract_overall(features, feature_list, train_folds, val_folds, test_folds)

### Hyperparameter tuning

In [ ]:
param_grid = {'penalty': ["elasticnet"],
              'C': [.1, 1, 10, 100],
              'l1_ratio': [0.25,0.5,0.75],
              'max_iter':[5000]}

param_grid = list(ParameterGrid(param_grid))

print("Logistic regression")
tune(X_train_list, Y_train_list, X_val_list, Y_val_list, param_grid, "log", balanced_accuracy=balanced_accuracy)
    

### Best params run that maximise AUROC (in this case...)

c=100, l1_ratio= .25, max_iter= 5000 penalty=elasticnet

In [ ]:
# for split in splits
train_scores_log = []

test_scores_log_auroc = []
test_scores_log_acc = []
test_scores_log_plr = []

final_test_files = []
final_preds_log = []

log_feat_importances_list = []

for fold in range(5):

    for file in test_folds[fold]:
        final_test_files.append(file)

    X_train = X_train_list[fold]
    Y_train = Y_train_list[fold]
   
    X_val = X_val_list[fold]
    Y_val = Y_val_list[fold]

    X_train = np.vstack((X_train, X_val))
    Y_train = np.concatenate((Y_train, Y_val))

    X_test = X_test_list[fold]
    Y_test = Y_test_list[fold]

    # Logistic regression model 
    pipeline = Pipeline([
        ('scaler', StandardScaler()),     
        ('logistic_regression', LogisticRegression(C=100, l1_ratio=0.25, max_iter=5000, penalty="elasticnet", solver="saga"))
    ])

    log_feat_importances, _, _ = fit_and_predict(pipeline, X_train, Y_train, X_test, Y_test, train_scores_log, test_scores_log_auroc, test_scores_log_acc, test_scores_log_plr, final_preds_log, balanced_accuracy=balanced_accuracy)
    log_feat_importances_list.append(log_feat_importances)


log_feat_importances, log_feat_importances_names = extract_importances(feature_list, log_feat_importances_list)

print("\n\n---- Logistic Regression\n")
print_results(final_test_files, final_preds_log, train_scores_log, test_scores_log_auroc, test_scores_log_acc, test_scores_log_plr, log_feat_importances_names, log_feat_importances)

## Violin plots

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,5))
mapping = {0:"No-response", 1: "Any-response"}
print(features.columns)
features_plot = features.with_columns(pl.col("gt").map_dict(mapping).alias("gt"))
for id, feature in enumerate(["cell_length","cell_perimeter"]):
    sns.violinplot(data=features_plot, x="gt", y=feature, ax=ax[id], palette={"No-response": "#1E88E5", "Any-response":"#1E88E5"}, hue="gt", legend=False, order=["No-response", "Any-response"])
    ax[id].set_xlabel('')
    ax[id].ticklabel_format(style="sci", axis="y", scilimits=(-3,3), useMathText=True)
#plt.savefig("../output/simple_classification/violin_plot_length_perimeter.svg")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,5))
mapping = {0:"No-response", 1: "Any-response"}
print(features.columns)
features_plot = features.with_columns(pl.col("gt").map_dict(mapping).alias("gt"))
for id, feature in enumerate(["cell_rgyration", "cell_area",]):
    sns.violinplot(data=features_plot, x="gt", y=feature, ax=ax[id], palette={"No-response": "#1E88E5", "Any-response":"#1E88E5"}, hue="gt", legend=False, order=["No-response", "Any-response"])
    ax[id].set_xlabel('')
    ax[id].ticklabel_format(style="sci", axis="y", scilimits=(-3,3), useMathText=True)
#plt.savefig("../output/simple_classification/violin_plot_rgyration_area.svg")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(16,5))
mapping = {0:"No-response", 1: "Any-response"}
print(features.columns)
features_plot = features.with_columns(pl.col("gt").map_dict(mapping).alias("gt"))
for id, feature in enumerate(["cell_linearity", "cell_planarity"]):
    sns.violinplot(data=features_plot, x="gt", y=feature, ax=ax[id], palette={"No-response": "#1E88E5", "Any-response":"#1E88E5"}, hue="gt", legend=False, order=["No-response", "Any-response"])
    ax[id].set_xlabel('')
    ax[id].ticklabel_format(style="sci", axis="y", scilimits=(-3,3), useMathText=True)
#plt.savefig("../output/simple_classification/violin_plot_linearity_planarity.svg")